# 07 - Final Demo: SeismoFinance Buy/Hold/Sell Prototype

Loads the trained LSTM (`models/lstm_model.keras`) and the fitted scaler, builds the most recent input window from the processed dataset, and turns the model output into a readable **SELL / HOLD / BUY** signal using `src/signals.py`.

If the trained model is not present, the notebook falls back to demo probabilities so it still renders.

> Academic decision-support prototype - **not financial advice.**

## 1. Setup & project modules

In [1]:
import os
import sys
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd

# Detect project root (works whether run from notebooks/ or repo root).
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.config import MODEL_DATASET_PATH, LSTM_MODEL_PATH, SCALER_PATH, CLASS_NAMES
from src.signals import probabilities_to_signal, format_signal_output
print("Project modules loaded.")
print("Class meaning:", CLASS_NAMES)


Project modules loaded.
Class meaning: {0: 'SELL', 1: 'HOLD', 2: 'BUY'}


## 2. Load the processed dataset

In [2]:
df = pd.read_csv(MODEL_DATASET_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["ticker", "date"]).reset_index(drop=True)
print("Dataset loaded:", df.shape)
print("Ticker:", df["ticker"].unique().tolist(), "| latest date:", df["date"].max().date())
df.tail(3)


Dataset loaded: (3389, 35)
Ticker: ['8766.T'] | latest date: 2023-12-28


,date,ticker,close,return,market_return,abnormal_return,quake_count_1d,max_magnitude_1d,avg_magnitude_1d,avg_depth_1d,...,return_lag_5,rolling_return_5,rolling_return_10,volatility_5,volatility_10,volatility_20,volume_change,target_return_next_day,target_abnormal_return_next_day,target_signal
3386,2023-12-26,8766.T,475.160538,-0.000170,-0.010803,0.010633,1.0,6.759047,6.759047,46.835127,...,0.006755,0.002057,0.000683,0.006709,0.006231,0.008084,-0.819958,0.010231,0.019691,2
3387,2023-12-27,8766.T,480.021767,0.010231,-0.009460,0.019691,1.0,6.478878,6.478878,505.361958,...,-0.007140,0.005531,0.001629,0.005048,0.006925,0.008121,5.153433,-0.009224,-0.019630,0
3388,2023-12-28,8766.T,475.594273,-0.009224,0.010407,-0.019630,0.0,0.000000,0.000000,0.000000,...,0.005349,0.002617,0.000930,0.008323,0.007671,0.008567,-0.526161,0.006094,0.012527,2


## 3. Feature columns (must match training)

In [3]:
# Must match EXACTLY the 30 feature columns (and order) used to train the model in 06.
feature_cols = [
    "close", "return", "market_return", "abnormal_return",
    "quake_count_1d", "max_magnitude_1d", "avg_magnitude_1d",
    "avg_depth_1d", "min_depth_1d", "energy_sum_1d", "energy_max_1d",
    "quake_count_3d", "max_magnitude_3d", "energy_sum_3d",
    "min_distance_tokyo_km", "min_distance_osaka_km", "min_distance_sendai_km",
    "min_distance_fukushima_km", "min_distance_major_city_km", "days_since_last_quake",
    "return_lag_1", "return_lag_2", "return_lag_3", "return_lag_5",
    "rolling_return_5", "rolling_return_10",
    "volatility_5", "volatility_10", "volatility_20",
    "volume_change",
]

# Keep only fully valid rows (same cleaning as training).
df_valid = df.dropna(subset=feature_cols).reset_index(drop=True)
print(f"{len(df_valid)} usable rows ({len(feature_cols)} features).")


3382 usable rows (30 features).


## 4. Load the trained model & scaler

In [4]:
import joblib
import tensorflow as tf

if LSTM_MODEL_PATH.exists() and SCALER_PATH.exists():
    model  = tf.keras.models.load_model(LSTM_MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)
    SEQ_LEN = model.input_shape[1]          # sequence length the model was trained with
    MODEL_READY = True
    print(f"Loaded model (expects sequences of length {SEQ_LEN}) and scaler.")
else:
    MODEL_READY = False
    print("Trained model/scaler not found - run notebook 06 first.")
    print("Expected:", LSTM_MODEL_PATH, "and", SCALER_PATH)


Loaded model (expects sequences of length 10) and scaler.


## 5. Predict the latest Buy/Hold/Sell signal

In [5]:
def predict_signal(history_df):
    """Take the most recent SEQ_LEN rows and return a Buy/Hold/Sell signal dict."""
    window = history_df[feature_cols].tail(SEQ_LEN)
    if len(window) < SEQ_LEN:
        raise ValueError(f"Need at least {SEQ_LEN} rows, got {len(window)}.")
    X = scaler.transform(window).reshape(1, SEQ_LEN, len(feature_cols))
    probs = model.predict(X, verbose=0)[0]      # softmax order = [SELL, HOLD, BUY]
    return probabilities_to_signal(probs)

if MODEL_READY:
    as_of = df_valid["date"].iloc[-1].date()
    result = predict_signal(df_valid)
    print(f"Prediction for the trading day after {as_of} (ticker {df_valid['ticker'].iloc[-1]}):\n")
    print(format_signal_output(result))
else:
    # Fallback so the notebook still renders without a trained model.
    demo_probabilities = np.array([0.42, 0.37, 0.21])
    print("[demo fallback - no trained model present]\n")
    print(format_signal_output(probabilities_to_signal(demo_probabilities)))


Prediction for the trading day after 2023-12-27 (ticker 8766.T):

SELL probability: 30.68%
HOLD probability: 27.86%
BUY probability: 41.46%
Final prototype signal: BUY


## 6. Signals for the last few trading days

In [6]:
# Show the signal the model would have produced for each of the last few trading days.
if MODEL_READY:
    rows = []
    for end in range(len(df_valid) - 5, len(df_valid)):
        sub = df_valid.iloc[: end + 1]
        r = predict_signal(sub)
        rows.append({
            "as_of_date": df_valid["date"].iloc[end].date(),
            "signal": r["final_signal"],
            "P(SELL)": round(r["sell_probability"], 3),
            "P(HOLD)": round(r["hold_probability"], 3),
            "P(BUY)":  round(r["buy_probability"], 3),
        })
    display(pd.DataFrame(rows))
else:
    print("No trained model loaded - skipping recent-days table.")


,as_of_date,signal,P(SELL),P(HOLD),P(BUY)
0,2023-12-21,BUY,0.310,0.290,0.399
1,2023-12-22,BUY,0.311,0.291,0.398
2,2023-12-25,BUY,0.296,0.305,0.398
3,2023-12-26,BUY,0.299,0.295,0.406
4,2023-12-27,BUY,0.307,0.279,0.415


## Interpretation

The final output is a prototype signal derived from the predicted next-day reaction:

- **SELL** - the model expects a negative abnormal return.
- **HOLD** - the model expects a roughly neutral reaction.
- **BUY** - the model expects a positive abnormal return.

As shown in notebook 06, the model's accuracy is only modestly above chance, so these signals are illustrative of the pipeline rather than reliable trading advice. This is an academic decision-support prototype and should not be interpreted as financial advice.